# Lab 11: MCMC and the 2D Ising model

A spin interacts with its four neighbors. We'll use that local rule to sample a whole lattice, then see what changes when we raise the temperature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "font.size": 13,
    "figure.figsize": (6.2, 3.5),
    "figure.dpi": 100,
    "figure.constrained_layout.use": True,
})
J = 1.0
KB = 1.0

We'll use NumPy for the lattice and Matplotlib for the plots. Run the cells in order; the later simulations use the functions we define along the way.

## Put the spins on a grid

Each entry is $+1$ or $-1$. Here $N$ is the side length, so the lattice contains $N^2$ spins. This is the configuration from lecture.

In [ ]:
N = 10
spins = np.ones((N, N), dtype=int)
spins[:5, :] = -1
spins[1, 3] = 1

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(spins, cmap="RdYlBu", vmin=-1, vmax=1)
ax.set_xticks(np.arange(-0.5, N, 1), minor=True)
ax.set_yticks(np.arange(-0.5, N, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.3)
ax.set(title="Blue: +1     Red: −1", xlabel="Column", ylabel="Row")
plt.show()

## Count each bond once

$$E=-J\sum_{\langle ij\rangle}\sigma_i\sigma_j.$$

Aligned neighbors contribute $-J$; opposite neighbors contribute $+J$. The edges wrap around. Summing all four neighbors counts every bond twice, so we divide by two.

In [ ]:
def total_energy(spins):
    n = spins.shape[0]
    energy = 0.0
    for i in range(n):
        for j in range(n):
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                neighbor = spins[(i + di) % n, (j + dj) % n]
                energy -= J * spins[i, j] * neighbor
    return energy / 2

print("Energy:", total_energy(spins))

For this configuration, the energy is −152 with $J=1$. The `% n` keeps the neighbor index inside the lattice, including at the edges.

## Temperature changes the probabilities

$$p(\sigma)=\frac{1}{Z}e^{-E(\sigma)/(k_{\mathrm B}T)}.$$

Lower-energy configurations are more likely. Raising the temperature makes an energy increase less costly.

We'll compare two configurations that differ by one spin. Their probability ratio does not depend on $Z$:

$$\frac{p(\sigma')}{p(\sigma)}=e^{-\Delta E/(k_{\mathrm B}T)}.$$

## Flip one spin

Only the four bonds touching that spin change. Since $\Delta\sigma_i=-2\sigma_i$,

$$\Delta E=2J\sigma_i\sum_{j\in\mathrm{neighbors}(i)}\sigma_j.$$

We can calculate this without recomputing the energy of the whole lattice.

In [ ]:
def flip_energy(spins, i, j):
    n = spins.shape[0]
    neighbors = (spins[(i - 1) % n, j] + spins[(i + 1) % n, j]
                 + spins[i, (j - 1) % n] + spins[i, (j + 1) % n])
    return 2 * J * spins[i, j] * neighbors

print("Flip the isolated +1 spin:", flip_energy(spins, 1, 3))

## Accept or reject

We choose a site uniformly, so proposing the reverse flip is equally likely.

- **Energy falls or stays the same:** accept.
- **Energy rises:** accept with probability $e^{-\Delta E/(k_{\mathrm B}T)}$.

A rejected proposal leaves the lattice unchanged. It still counts as a step.

In [ ]:
def metropolis_step(spins, temperature, rng):
    i, j = rng.integers(spins.shape[0], size=2)
    delta_energy = flip_energy(spins, i, j)
    if delta_energy <= 0 or rng.random() < np.exp(-delta_energy / (KB * temperature)):
        spins[i, j] *= -1
        return 1
    return 0

## Magnetization

The total magnetization is $M$. We'll plot the magnetization per spin, $m$:

$$M=\sum_{i\in\Lambda}\sigma_i,\qquad m=\frac{M}{N^2}.$$

All spins up gives $m=1$; all down gives $m=-1$. A nearly equal mix gives $m\approx0$.

In Python, `spins.mean()` is $m$.

## Count time in sweeps

One **sweep** is $N^2$ attempted flips. Sites are chosen randomly, so some can be chosen more than once.

We'll start with all spins up and record $m$ after each sweep, whether the proposals were accepted or rejected. The seed makes a run reproducible.

In [ ]:
def run_ising(temperature, n, sweeps, seed):
    rng = np.random.default_rng(seed)
    spins = np.ones((n, n), dtype=int)
    magnetization = np.empty(sweeps + 1)
    magnetization[0] = spins.mean()
    accepted = 0
    for sweep in range(1, sweeps + 1):
        for _ in range(n * n):
            accepted += metropolis_step(spins, temperature, rng)
        magnetization[sweep] = spins.mean()
    return spins, magnetization, accepted / (sweeps * n * n)

## Run at $T=2$

Use $J=k_{\mathrm B}=1$ and a $24\times24$ lattice to keep the live calculation short. The acceptance rate is the fraction of proposed flips that changed a spin.

In [ ]:
N = 24
SWEEPS = 400
SEED = 142
T = 2.0

final_spins, m_values, acceptance = run_ising(T, N, SWEEPS, SEED)
print(f"Attempted flips: {SWEEPS * N * N:,}")
print(f"Acceptance rate: {acceptance:.3f}")

## Look at the lattice

This is one configuration at the end of the run. We'll also look at the history of the magnetization, since a final picture doesn't tell us how the chain got there.

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.imshow(final_spins, cmap="RdYlBu", vmin=-1, vmax=1)
ax.set(title=f"T = {T}, after {SWEEPS} sweeps",
       xlabel="Column", ylabel="Row")
plt.show()

## Follow the magnetization

The chain starts at $m=1$. We'll set aside the first 100 sweeps as **burn-in** for this demonstration. That cutoff is a choice to inspect, not a guarantee that the chain has equilibrated.

In [ ]:
BURNIN = 100
sweep_axis = np.arange(SWEEPS + 1)

fig, ax = plt.subplots()
ax.plot(sweep_axis, m_values, color="#007c91")
ax.axvspan(0, BURNIN, color="gray", alpha=0.2, label="Burn-in")
ax.axvline(BURNIN, color="gray", linestyle="--")
ax.set(xlabel="Sweeps", ylabel="Magnetization per spin m",
       title=f"T = {T}", ylim=(-1.05, 1.05))
ax.legend()
plt.show()

## After burn-in

We'll summarize the values after the cutoff. The standard deviation measures how much $m$ fluctuates; it is **not the uncertainty on the mean**. Consecutive measurements can still be correlated.

In [ ]:
samples = m_values[BURNIN + 1:]
print(f"Mean m: {samples.mean():.3f}")
print(f"Standard deviation of m: {samples.std():.3f}")
print(f"Mean |m|: {np.abs(samples).mean():.3f}")

$\langle|m|\rangle$ measures the amount of alignment without favoring up over down. It differs from $|\langle m\rangle|$: a chain can switch between positive and negative magnetization.

## Raise the temperature

For the infinite square lattice at zero field,

$$T_{\mathrm c}=\frac{2J}{k_{\mathrm B}\ln(1+\sqrt2)}\approx2.269.$$

Below this temperature, the infinite system has spontaneous magnetization. Our finite lattice shows a rounded version of this change.

## Compare three temperatures

We'll run below, near, and above $T_{\mathrm c}$ with the same lattice size and number of sweeps. Each run starts from all spins up.

In [ ]:
temperatures = [1.5, 2.269, 3.5]
runs = {}
for index, temperature in enumerate(temperatures):
    runs[temperature] = run_ising(temperature, N, SWEEPS, SEED + index)
    _, history, rate = runs[temperature]
    mean_abs_m = np.abs(history[BURNIN + 1:]).mean()
    print(f"T = {temperature:.3f}: mean |m| = {mean_abs_m:.3f}, "
          f"acceptance = {rate:.3f}")

## The final configurations

At low temperature, large regions tend to align. At high temperature, energy-increasing flips are accepted more often and that order breaks up.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(7.2, 3.0))
for ax, temperature in zip(axes, temperatures):
    lattice, _, _ = runs[temperature]
    ax.imshow(lattice, cmap="RdYlBu", vmin=-1, vmax=1)
    ax.set(title=f"T = {temperature:g}", xticks=[], yticks=[])
plt.show()

## The histories matter too

Near the transition, relaxation can be slow. These short runs show temperature dependence on a finite lattice; they do not locate the critical temperature precisely.

In [ ]:
fig, ax = plt.subplots()
for temperature in temperatures:
    _, history, _ = runs[temperature]
    ax.plot(sweep_axis, history, label=f"T = {temperature:g}")
ax.axvspan(0, BURNIN, color="gray", alpha=0.2)
ax.set(xlabel="Sweeps", ylabel="Magnetization per spin m",
       ylim=(-1.05, 1.05))
ax.legend()
plt.show()

## From a local flip to a thermal average

We used four neighbors to calculate $\Delta E$, accepted or rejected a flip, and repeated that rule across the lattice.

The temperature sets how readily the system accepts an energy increase. The magnetization shows the resulting order, while its history tells us what one final picture leaves out.